# Chapter 04: Ray Tracing

Source orientation: printed pages 79-96; physical PDF pages 96-113. The source span covers image-order rendering, perspective, camera-frame ray generation, ray-sphere and ray-triangle intersections, closest-hit selection, shading inputs, shadows, mirror reflection, and the small numerical offsets used to keep secondary rays from hitting the surface that spawned them. This notebook uses that span only to choose the coverage. The prose, code, diagrams, and checks here are original course material.

## Chapter goal

Build a small ray tracer that is inspectable at every geometric step. By the end, a viewing ray is not an abstract line in a diagram: it is a tested object with an origin, a direction, a valid interval, a closest hit record, surface data for shading, and carefully offset secondary rays for shadows and mirror reflection.

## Translation guide: computational ray tracing

| Book idea | Computational object in this notebook | What can go wrong |
| --- | --- | --- |
| Image-order rendering | outer loop over pixel centers; one primary ray per pixel | pixel-center convention or image axes flipped |
| Perspective camera | eye point plus orthonormal camera frame `u, v, w`; each pixel direction fans through an image plane | wrong sign for `w`, unnormalized frame, or omitted half-pixel shift |
| Ray parameter | `p(t) = origin + t * direction` with an active interval `[t0, t1]` | accepting hits behind the eye or beyond the current closest hit |
| Sphere intersection | quadratic roots sorted by increasing `t` | unstable roots, accepting the exit point when the entry is valid, missing tangency |
| Triangle intersection | linear solve for `t` and barycentric coordinates | treating a plane hit outside the triangle as a surface hit |
| Closest visible surface | hit record with smallest valid positive `t` | overwriting a near hit with a later far hit |
| Shading inputs | hit point `x`, unit normal `n`, light direction `l`, view direction `v` | using unnormalized vectors or the wrong direction for `v` |
| Shadows | secondary ray from `x` toward the light over a finite interval | self-intersection acne when `t0 = 0`; missing close blockers when epsilon is too large |
| Mirror reflection | recursive ray in `d - 2(d dot n)n` direction with a depth limit | infinite recursion or a reflection ray that immediately re-hits its source |

## Visual storyboard

1. A dataflow graph shows the dependency chain from camera pixels to final color and highlights where intervals and hit records enter.
2. A camera-ray grid compares the image-plane pixel convention with perspective ray directions, with an interactive 3D HTML fan for inspection.
3. Sphere and triangle intersection diagnostics expose the algebraic tests that decide whether a ray really hit a surface.
4. A closest-hit interval diagram shows why the scene intersection loop shrinks `t1` after every nearer hit.
5. Shading, shadow, reflection, and epsilon diagrams connect the visible hit record to the secondary rays that make ray tracing useful.
6. The applied lab renders a small scene twice, changing reflection depth, so the recursive contribution is visible and checked.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import plotly.graph_objects as go
import sympy as sp

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the FCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (
    assert_artifacts,
    book_relative,
    display_artifact,
    save_json,
    save_matplotlib,
    save_plotly_html,
    save_table_csv,
)
from utils.notebook_checks import assert_nonblank_image
from utils.plotting import PALETTE, add_note, close, style_axis

CHAPTER = 4
TOPIC = "chapter-04"
PRINTED_PAGES = "79-96"
PDF_PAGES = "96-113"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
for kind in ["figures", "html", "checks", "tables", "data"]:
    (ARTIFACT_ROOT / kind).mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(20260512)
image_paths: list[Path] = []
html_paths: list[Path] = []
table_paths: list[Path] = []
check_paths: list[Path] = []
checks: dict[str, object] = {
    "chapter": CHAPTER,
    "printed_pages": PRINTED_PAGES,
    "pdf_pages": PDF_PAGES,
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#fbfcfe",
    "axes.edgecolor": "#b6c0ca",
    "font.size": 9,
})


def unit(v: np.ndarray, *, eps: float = 1e-12) -> np.ndarray:
    arr = np.asarray(v, dtype=float)
    norm = np.linalg.norm(arr)
    if norm < eps:
        raise ValueError("cannot normalize a near-zero vector")
    return arr / norm


@dataclass(frozen=True)
class Ray:
    origin: np.ndarray
    direction: np.ndarray

    def at(self, t: float) -> np.ndarray:
        return self.origin + t * self.direction


@dataclass(frozen=True)
class Hit:
    t: float
    point: np.ndarray
    normal: np.ndarray
    name: str
    color: np.ndarray
    mirror: np.ndarray


@dataclass(frozen=True)
class CameraFrame:
    eye: np.ndarray
    u: np.ndarray
    v: np.ndarray
    w: np.ndarray
    left: float
    right: float
    bottom: float
    top: float
    distance: float


def make_camera(eye, look_at, up, *, left=-1.6, right=1.6, bottom=-0.9, top=0.9, distance=1.2) -> CameraFrame:
    eye = np.asarray(eye, dtype=float)
    gaze = unit(np.asarray(look_at, dtype=float) - eye)
    w = -gaze
    u = unit(np.cross(np.asarray(up, dtype=float), w))
    v = np.cross(w, u)
    return CameraFrame(eye, u, v, w, float(left), float(right), float(bottom), float(top), float(distance))


def pixel_uv(i: int, j: int, nx: int, ny: int, cam: CameraFrame) -> tuple[float, float]:
    s = cam.left + (cam.right - cam.left) * (i + 0.5) / nx
    t_coord = cam.bottom + (cam.top - cam.bottom) * (j + 0.5) / ny
    return float(s), float(t_coord)


def perspective_ray(i: int, j: int, nx: int, ny: int, cam: CameraFrame, *, normalize_direction: bool = True) -> Ray:
    s, t_coord = pixel_uv(i, j, nx, ny, cam)
    direction = -cam.distance * cam.w + s * cam.u + t_coord * cam.v
    if normalize_direction:
        direction = unit(direction)
    return Ray(cam.eye.copy(), direction)


def orthographic_ray(i: int, j: int, nx: int, ny: int, cam: CameraFrame) -> Ray:
    s, t_coord = pixel_uv(i, j, nx, ny, cam)
    origin = cam.eye + s * cam.u + t_coord * cam.v
    return Ray(origin, -cam.w)


def reflect(direction: np.ndarray, normal: np.ndarray) -> np.ndarray:
    d = unit(direction)
    n = unit(normal)
    return d - 2.0 * np.dot(d, n) * n


## 1. Ray Tracing Dataflow

The basic algorithm has only three large verbs: generate a viewing ray, intersect it with the scene, and shade the closest valid hit. The important engineering detail is that the verbs pass structured data rather than just booleans. Ray generation passes an origin, direction, and pixel identity. Intersection returns a hit record with `t`, `x`, `n`, material data, and the object name. Shading may then launch shadow and reflection rays, each with a deliberately chosen interval.

The graph below is a proof scaffold for implementation: every edge is a dependency the code must honor. In particular, the ray interval flows into intersection, closest-hit logic, shadow testing, and reflection offsets. A renderer with beautiful formulas but careless intervals will still produce wrong images.


In [ ]:
storyboard = {
    "chapter_goal": "Make basic ray tracing inspectable from camera rays through secondary rays.",
    "source_span_read": "Printed pages 79-96; physical PDF pages 96-113.",
    "concept_inventory": [
        "image-order rendering versus object-order rendering",
        "linear perspective and orthographic camera rays",
        "camera frame u, v, w and half-pixel image-plane coordinates",
        "parametric ray p(t) and valid t intervals",
        "sphere roots from a quadratic discriminant",
        "triangle hits from barycentric coordinates and a 3x3 solve",
        "closest-hit records for grouped surfaces",
        "shading inputs x, n, l, and v",
        "shadow rays with epsilon offsets",
        "mirror reflection rays with depth limits",
    ],
    "library_routing": [
        {"concept": "camera ray fan", "library": "Plotly", "why": "3D fan is easier to rotate and inspect in HTML."},
        {"concept": "static intersection diagnostics", "library": "Matplotlib", "why": "2D slices, root plots, and hit maps need durable labeled PNGs."},
        {"concept": "implementation dependency scaffold", "library": "NetworkX", "why": "the chapter algorithm is a data dependency graph."},
        {"concept": "reflection invariant", "library": "SymPy", "why": "exact symbolic check shows reflection preserves length for unit normals."},
        {"concept": "ray/object experiments", "library": "NumPy", "why": "small vector computations make t intervals and normals explicit."},
    ],
    "visual_sequence": [
        "ray-tracing-dataflow-dependencies.png",
        "perspective-camera-ray-grid.png",
        "perspective-camera-ray-fan.html",
        "sphere-quadratic-intersection-diagnostics.png",
        "triangle-barycentric-hit-map.png",
        "closest-hit-interval-shrink.png",
        "shading-shadow-reflection-rays.png",
        "epsilon-shadow-acne-pitfall.png",
        "reflection-depth-comparison.png",
        "mini-ray-traced-scene.png",
    ],
}
storyboard_path = save_json(storyboard, TOPIC, "visual-storyboard.json")
check_paths.append(storyboard_path)

nodes = [
    "pixel center\n(i + 1/2, j + 1/2)",
    "camera frame\ne, u, v, w",
    "primary ray\no, d, [t0, t1]",
    "surface tests\nsphere / triangle",
    "closest hit\nmin valid t",
    "hit record\nx, n, material",
    "direct light\nl, v, n dot l",
    "shadow ray\n[x + epsilon l, light]",
    "reflection ray\nd - 2(d dot n)n",
    "pixel color\nlocal + recursive",
]
edges = [
    (nodes[0], nodes[2]), (nodes[1], nodes[2]), (nodes[2], nodes[3]),
    (nodes[3], nodes[4]), (nodes[4], nodes[5]), (nodes[5], nodes[6]),
    (nodes[5], nodes[7]), (nodes[5], nodes[8]), (nodes[6], nodes[9]),
    (nodes[7], nodes[9]), (nodes[8], nodes[9]),
]
G = nx.DiGraph()
G.add_edges_from(edges)
pos = {
    nodes[0]: (0.0, 1.0), nodes[1]: (0.0, -0.3), nodes[2]: (1.4, 0.35),
    nodes[3]: (2.75, 0.35), nodes[4]: (4.0, 0.35), nodes[5]: (5.25, 0.35),
    nodes[6]: (6.6, 1.2), nodes[7]: (6.6, 0.35), nodes[8]: (6.6, -0.55),
    nodes[9]: (8.0, 0.35),
}
fig, ax = plt.subplots(figsize=(12, 4.6))
node_colors = ["#eaf2ff" if "ray" in n else "#f5fbf7" if "hit" in n else "#fff7e6" for n in G.nodes]
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=16, width=1.8, edge_color="#6b7280")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=3100, edgecolors="#334155", linewidths=1.0)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_color="#111827")
ax.set_title("Ray tracing data dependencies", fontsize=13, color=PALETTE["ink"])
ax.set_axis_off()
add_note(ax, "Inspection target: intervals and hit records must survive every stage.")
dataflow_path = save_matplotlib(fig, TOPIC, "ray-tracing-dataflow-dependencies.png")
close(fig)
image_paths.append(dataflow_path)
checks["dataflow_nodes"] = G.number_of_nodes()
checks["dataflow_edges"] = G.number_of_edges()
dataflow_path


## 2. Perspective Camera Rays

The camera frame is an orthonormal basis attached to the eye. In this notebook `w` points backward, so the view direction is `-w`. Pixel centers first become image-plane coordinates by adding half a pixel before scaling into `[left, right] x [bottom, top]`. An orthographic ray changes origin from pixel to pixel while keeping direction fixed. A perspective ray keeps the origin at the eye and changes direction through the image plane.

The static figure lets you inspect two easy failure modes: forgetting the half-pixel shift moves samples to pixel corners, and using `+w` instead of `-w` sends perspective rays behind the camera. The HTML artifact stores the same fan in 3D so the right-handed `u, v, w` convention can be rotated.


In [ ]:
cam = make_camera(
    eye=np.array([0.0, 0.0, 2.8]),
    look_at=np.array([0.0, 0.0, 0.0]),
    up=np.array([0.0, 1.0, 0.0]),
    left=-1.6,
    right=1.6,
    bottom=-0.9,
    top=0.9,
    distance=1.4,
)
nx_pixels, ny_pixels = 9, 5
uv_grid = np.array([pixel_uv(i, j, nx_pixels, ny_pixels, cam) for j in range(ny_pixels) for i in range(nx_pixels)])
center_ray = perspective_ray(nx_pixels // 2, ny_pixels // 2, nx_pixels, ny_pixels, cam)
center_alignment = float(np.dot(center_ray.direction, -cam.w))
checks["camera_frame_orthonormal_error"] = float(abs(np.dot(cam.u, cam.v)) + abs(np.dot(cam.u, cam.w)) + abs(np.dot(cam.v, cam.w)))
checks["center_ray_alignment_with_view"] = center_alignment
checks["pixel_center_uv"] = list(map(float, pixel_uv(nx_pixels // 2, ny_pixels // 2, nx_pixels, ny_pixels, cam)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
ax = axes[0]
ax.scatter(uv_grid[:, 0], uv_grid[:, 1], s=32, color=PALETTE["blue"], zorder=3, label="pixel centers")
for x in np.linspace(cam.left, cam.right, nx_pixels + 1):
    ax.plot([x, x], [cam.bottom, cam.top], color="#d6dee8", lw=0.8)
for y in np.linspace(cam.bottom, cam.top, ny_pixels + 1):
    ax.plot([cam.left, cam.right], [y, y], color="#d6dee8", lw=0.8)
ax.scatter([0], [0], marker="+", s=130, color=PALETTE["red"], label="optical axis")
style_axis(ax, "image plane coordinates", equal=True, xlabel="u coordinate", ylabel="v coordinate")
ax.legend(loc="upper right", fontsize=8)
add_note(ax, "The samples are centered inside pixels, not on grid lines.")

ax = axes[1]
plane_z = cam.eye[2] - cam.distance
ax.scatter([cam.eye[0]], [cam.eye[2]], color=PALETTE["red"], s=70, label="eye")
ax.plot([cam.left, cam.right], [plane_z, plane_z], color=PALETTE["gray"], lw=2.0, label="image plane")
for i in range(nx_pixels):
    ray = perspective_ray(i, ny_pixels // 2, nx_pixels, ny_pixels, cam, normalize_direction=False)
    end = cam.eye + 2.9 * unit(ray.direction)
    color = PALETTE["blue"] if i != nx_pixels // 2 else PALETTE["red"]
    ax.plot([cam.eye[0], end[0]], [cam.eye[2], end[2]], color=color, lw=1.2)
for i in [0, nx_pixels // 2, nx_pixels - 1]:
    ray = orthographic_ray(i, ny_pixels // 2, nx_pixels, ny_pixels, cam)
    start = ray.origin
    end = start + 2.2 * ray.direction
    ax.plot([start[0], end[0]], [start[2], end[2]], color=PALETTE["green"], lw=1.0, ls="--")
style_axis(ax, "perspective fan vs orthographic parallels", equal=True, xlabel="world x", ylabel="world z")
ax.invert_yaxis()
ax.legend(loc="lower right", fontsize=8)
add_note(ax, "Perspective: same origin, different directions. Orthographic: different origins, same direction.")

camera_grid_path = save_matplotlib(fig, TOPIC, "perspective-camera-ray-grid.png")
close(fig)
image_paths.append(camera_grid_path)

ray_lines = []
for j in range(ny_pixels):
    for i in range(nx_pixels):
        s, t_coord = pixel_uv(i, j, nx_pixels, ny_pixels, cam)
        point_on_plane = cam.eye - cam.distance * cam.w + s * cam.u + t_coord * cam.v
        ray_lines.append(go.Scatter3d(
            x=[cam.eye[0], point_on_plane[0]],
            y=[cam.eye[1], point_on_plane[1]],
            z=[cam.eye[2], point_on_plane[2]],
            mode="lines",
            line={"color": "rgba(47,111,187,0.35)", "width": 3},
            showlegend=False,
            hoverinfo="skip",
        ))
plane_corners = np.array([
    cam.eye - cam.distance * cam.w + cam.left * cam.u + cam.bottom * cam.v,
    cam.eye - cam.distance * cam.w + cam.right * cam.u + cam.bottom * cam.v,
    cam.eye - cam.distance * cam.w + cam.right * cam.u + cam.top * cam.v,
    cam.eye - cam.distance * cam.w + cam.left * cam.u + cam.top * cam.v,
    cam.eye - cam.distance * cam.w + cam.left * cam.u + cam.bottom * cam.v,
])
fig3 = go.Figure(ray_lines)
fig3.add_trace(go.Scatter3d(x=plane_corners[:, 0], y=plane_corners[:, 1], z=plane_corners[:, 2], mode="lines", line={"color": "#1f2933", "width": 5}, name="image plane"))
fig3.add_trace(go.Scatter3d(x=[cam.eye[0]], y=[cam.eye[1]], z=[cam.eye[2]], mode="markers+text", marker={"size": 5, "color": "#c44e52"}, text=["eye"], textposition="top center", name="eye"))
for label, vec, color in [("u", cam.u, "#2f6fbb"), ("v", cam.v, "#2a9d8f"), ("-w", -cam.w, "#c44e52")]:
    end = cam.eye + 0.75 * vec
    fig3.add_trace(go.Scatter3d(x=[cam.eye[0], end[0]], y=[cam.eye[1], end[1]], z=[cam.eye[2], end[2]], mode="lines+text", line={"color": color, "width": 6}, text=["", label], textposition="top center", name=label))
fig3.update_layout(
    title="Perspective camera ray fan: rotate to inspect the camera frame",
    scene={"aspectmode": "data", "xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "z"},
    margin={"l": 0, "r": 0, "t": 40, "b": 0},
)
camera_html_path = save_plotly_html(fig3, TOPIC, "perspective-camera-ray-fan.html")
html_paths.append(camera_html_path)

checks["camera_center_ray_passes_optical_axis"] = bool(center_alignment > 0.999999)
checks["camera_corner_rays_differ"] = bool(np.linalg.norm(perspective_ray(0, 0, nx_pixels, ny_pixels, cam).direction - perspective_ray(nx_pixels - 1, ny_pixels - 1, nx_pixels, ny_pixels, cam).direction) > 0.5)
(camera_grid_path, camera_html_path)


## 3. Ray-Object Intersections

A ray tracer does not project a full object first; it asks each object whether a particular ray meets it in the permitted interval. For a sphere, substituting the ray into the implicit equation produces a quadratic in `t`. The discriminant says whether the ray misses, grazes, or crosses the sphere. For a triangle, solving a 3-by-3 linear system gives both the ray parameter and barycentric coordinates; the plane hit only counts if the barycentric coordinates stay inside the triangle.

The two diagnostics below are intentionally small. They let you read the exact `t` roots and the exact inside/outside predicate that an implementation will later use in the image loop.


In [ ]:
def sphere_roots(ray: Ray, center: np.ndarray, radius: float, *, t0=0.0, t1=np.inf) -> tuple[float, float] | None:
    oc = ray.origin - center
    a = float(np.dot(ray.direction, ray.direction))
    b = 2.0 * float(np.dot(oc, ray.direction))
    c = float(np.dot(oc, oc) - radius * radius)
    disc = b * b - 4.0 * a * c
    if disc < 0:
        return None
    root_disc = math.sqrt(max(0.0, disc))
    q = -0.5 * (b + math.copysign(root_disc, b if b != 0 else 1.0))
    if abs(q) < 1e-15:
        roots = sorted([(-b - root_disc) / (2.0 * a), (-b + root_disc) / (2.0 * a)])
    else:
        roots = sorted([q / a, c / q])
    return (float(roots[0]), float(roots[1]))


def sphere_hit(ray: Ray, center: np.ndarray, radius: float, name="sphere", color=None, mirror=None, *, t0=1e-5, t1=np.inf) -> Hit | None:
    roots = sphere_roots(ray, center, radius, t0=t0, t1=t1)
    if roots is None:
        return None
    for root in roots:
        if t0 <= root <= t1:
            point = ray.at(root)
            normal = unit(point - center)
            return Hit(root, point, normal, name, np.array([0.45, 0.6, 0.9]) if color is None else np.asarray(color, dtype=float), np.zeros(3) if mirror is None else np.asarray(mirror, dtype=float))
    return None

sphere_center = np.array([0.0, 0.0, -4.0])
sphere_radius = 1.0
xs = np.linspace(-1.45, 1.45, 241)
root_near = np.full_like(xs, np.nan, dtype=float)
root_far = np.full_like(xs, np.nan, dtype=float)
discriminants = np.empty_like(xs)
for idx, x in enumerate(xs):
    ray = Ray(np.zeros(3), unit(np.array([x, 0.0, -1.0])))
    oc = ray.origin - sphere_center
    a = np.dot(ray.direction, ray.direction)
    b = 2.0 * np.dot(oc, ray.direction)
    c = np.dot(oc, oc) - sphere_radius**2
    discriminants[idx] = b * b - 4 * a * c
    roots = sphere_roots(ray, sphere_center, sphere_radius)
    if roots is not None:
        root_near[idx], root_far[idx] = roots

debug_ray = Ray(np.array([0.0, 0.0, 0.0]), np.array([0.0, 0.0, -1.0]))
debug_roots = sphere_roots(debug_ray, sphere_center, sphere_radius)
debug_hit = sphere_hit(debug_ray, sphere_center, sphere_radius, t0=0.0)
checks["sphere_center_roots"] = [float(debug_roots[0]), float(debug_roots[1])]
checks["sphere_center_hit_normal"] = debug_hit.normal.round(12).tolist()
checks["sphere_miss_count_in_sweep"] = int(np.count_nonzero(discriminants < 0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ax = axes[0]
ax.plot(xs, discriminants, color=PALETTE["blue"], lw=2.0)
ax.axhline(0, color=PALETTE["red"], lw=1.0)
style_axis(ax, "sphere discriminant by image-plane x", xlabel="x coordinate on image plane", ylabel="B^2 - 4AC")
add_note(ax, "positive: two roots; zero: tangent; negative: miss")

ax = axes[1]
ax.plot(xs, root_near, color=PALETTE["green"], lw=2.0, label="entry root")
ax.plot(xs, root_far, color=PALETTE["violet"], lw=2.0, label="exit root")
ax.axhline(0, color="#9aa6b2", lw=1.0)
style_axis(ax, "valid roots along the ray", xlabel="x coordinate on image plane", ylabel="ray parameter t")
ax.legend(fontsize=8)
add_note(ax, "The first valid root is the visible sphere hit.")

sphere_diag_path = save_matplotlib(fig, TOPIC, "sphere-quadratic-intersection-diagnostics.png")
close(fig)
image_paths.append(sphere_diag_path)
sphere_diag_path


In [ ]:
def triangle_hit(ray: Ray, a: np.ndarray, b: np.ndarray, c: np.ndarray, name="triangle", color=None, mirror=None, *, t0=1e-5, t1=np.inf) -> Hit | None:
    matrix = np.column_stack((b - a, c - a, -ray.direction))
    rhs = ray.origin - a
    det = float(np.linalg.det(matrix))
    if abs(det) < 1e-12:
        return None
    beta, gamma, t = np.linalg.solve(matrix, rhs)
    alpha = 1.0 - beta - gamma
    if t0 <= t <= t1 and alpha >= -1e-10 and beta >= -1e-10 and gamma >= -1e-10:
        point = ray.at(float(t))
        normal = unit(np.cross(b - a, c - a))
        return Hit(float(t), point, normal, name, np.array([0.92, 0.54, 0.26]) if color is None else np.asarray(color, dtype=float), np.zeros(3) if mirror is None else np.asarray(mirror, dtype=float))
    return None

tri_a = np.array([-1.1, -0.72, -3.0])
tri_b = np.array([1.0, -0.55, -3.0])
tri_c = np.array([-0.05, 0.85, -3.0])
probe_n = 95
xv = np.linspace(-1.35, 1.35, probe_n)
yv = np.linspace(-1.05, 1.05, probe_n)
hit_mask = np.zeros((probe_n, probe_n), dtype=bool)
beta_map = np.full((probe_n, probe_n), np.nan)
gamma_map = np.full((probe_n, probe_n), np.nan)
for jj, y in enumerate(yv):
    for ii, x in enumerate(xv):
        ray = Ray(np.zeros(3), unit(np.array([x, y, -1.0])))
        matrix = np.column_stack((tri_b - tri_a, tri_c - tri_a, -ray.direction))
        rhs = ray.origin - tri_a
        try:
            beta, gamma, t = np.linalg.solve(matrix, rhs)
        except np.linalg.LinAlgError:
            continue
        alpha = 1.0 - beta - gamma
        if t > 0 and min(alpha, beta, gamma) >= -1e-10:
            hit_mask[jj, ii] = True
            beta_map[jj, ii] = beta
            gamma_map[jj, ii] = gamma

center_ray_for_triangle = Ray(np.zeros(3), unit(np.array([0.0, -0.12, -1.0])))
center_hit = triangle_hit(center_ray_for_triangle, tri_a, tri_b, tri_c, t0=0.0)
beta_gamma_t = np.linalg.solve(np.column_stack((tri_b - tri_a, tri_c - tri_a, -center_ray_for_triangle.direction)), center_ray_for_triangle.origin - tri_a)
beta_hit, gamma_hit, t_hit = [float(v) for v in beta_gamma_t]
alpha_hit = 1.0 - beta_hit - gamma_hit
checks["triangle_debug_barycentric"] = {"alpha": alpha_hit, "beta": beta_hit, "gamma": gamma_hit, "sum": alpha_hit + beta_hit + gamma_hit}
checks["triangle_debug_t"] = t_hit
checks["triangle_hit_pixel_count"] = int(hit_mask.sum())
checks["triangle_plane_z_residual"] = float(abs(center_hit.point[2] + 3.0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.7))
ax = axes[0]
ax.imshow(hit_mask, origin="lower", extent=[xv.min(), xv.max(), yv.min(), yv.max()], cmap="Greens", alpha=0.78, aspect="equal")
projected = np.array([[tri_a[0] / -tri_a[2], tri_a[1] / -tri_a[2]], [tri_b[0] / -tri_b[2], tri_b[1] / -tri_b[2]], [tri_c[0] / -tri_c[2], tri_c[1] / -tri_c[2]], [tri_a[0] / -tri_a[2], tri_a[1] / -tri_a[2]]])
ax.plot(projected[:, 0], projected[:, 1], color=PALETTE["red"], lw=2.0, label="projected triangle")
ax.scatter([0], [-0.12], color=PALETTE["blue"], s=45, label="debug ray")
style_axis(ax, "rays whose plane hit lies inside the triangle", equal=True, xlabel="image-plane x", ylabel="image-plane y")
ax.legend(fontsize=8)
add_note(ax, "Inside means alpha, beta, and gamma are all nonnegative.")

ax = axes[1]
inside_beta = beta_map[hit_mask]
inside_gamma = gamma_map[hit_mask]
ax.scatter(inside_beta, inside_gamma, s=6, color=PALETTE["violet"], alpha=0.45)
ax.plot([0, 1, 0, 0], [0, 0, 1, 0], color=PALETTE["ink"], lw=1.5)
ax.scatter([beta_hit], [gamma_hit], color=PALETTE["red"], s=60, zorder=5)
style_axis(ax, "barycentric coordinates of accepted hits", equal=True, xlabel="beta", ylabel="gamma")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
add_note(ax, f"debug hit: alpha={alpha_hit:.2f}, beta={beta_hit:.2f}, gamma={gamma_hit:.2f}")

triangle_path = save_matplotlib(fig, TOPIC, "triangle-barycentric-hit-map.png")
close(fig)
image_paths.append(triangle_path)
triangle_path


## 4. Closest-Hit Logic And Ray Intervals

A scene can contain many surfaces, but a camera ray should report the first visible one. The simplest group algorithm keeps the nearest hit found so far and passes that value as the new `t1` to later surface tests. This is more than an optimization: it makes the visible-object rule explicit, and it prevents a farther object from overwriting a nearer one.

The ledger below records candidate hits in list order. The diagram shows the active ray interval shrinking after the first accepted hit. The same routine is tested with reversed object order to guard against a common implementation error.


In [ ]:
scene_spheres = [
    {"name": "rear blue sphere", "center": np.array([0.12, 0.0, -5.3]), "radius": 1.0, "color": np.array([0.34, 0.48, 0.88]), "mirror": np.array([0.05, 0.05, 0.08])},
    {"name": "front gold sphere", "center": np.array([-0.18, -0.08, -3.15]), "radius": 0.72, "color": np.array([0.95, 0.66, 0.22]), "mirror": np.array([0.15, 0.12, 0.05])},
    {"name": "small green sphere", "center": np.array([0.75, 0.25, -4.2]), "radius": 0.45, "color": np.array([0.35, 0.78, 0.52]), "mirror": np.array([0.02, 0.08, 0.04])},
]
probe_ray = Ray(np.array([0.0, 0.0, 0.0]), unit(np.array([-0.08, -0.02, -1.0])))


def first_sphere_hit(ray: Ray, spheres: list[dict], *, t0=1e-5, t1=np.inf) -> tuple[Hit | None, list[dict[str, object]]]:
    closest: Hit | None = None
    ledger: list[dict[str, object]] = []
    current_t1 = float(t1)
    for sphere in spheres:
        roots = sphere_roots(ray, sphere["center"], sphere["radius"])
        candidate_t = None
        accepted = False
        if roots is not None:
            for root in roots:
                if t0 <= root <= current_t1:
                    candidate_t = float(root)
                    point = ray.at(candidate_t)
                    closest = Hit(candidate_t, point, unit(point - sphere["center"]), sphere["name"], sphere["color"], sphere["mirror"])
                    current_t1 = candidate_t
                    accepted = True
                    break
            if candidate_t is None:
                candidate_t = next((float(root) for root in roots if root >= t0), None)
        ledger.append({
            "surface": sphere["name"],
            "roots": None if roots is None else [float(roots[0]), float(roots[1])],
            "candidate_t": candidate_t,
            "accepted_as_closest_so_far": accepted,
            "interval_t1_after_test": current_t1,
        })
    return closest, ledger

hit_forward, ledger_forward = first_sphere_hit(probe_ray, scene_spheres)
hit_reverse, ledger_reverse = first_sphere_hit(probe_ray, list(reversed(scene_spheres)))
checks["closest_hit_name"] = hit_forward.name
checks["closest_hit_t"] = float(hit_forward.t)
checks["closest_hit_order_invariant"] = bool(hit_forward.name == hit_reverse.name and abs(hit_forward.t - hit_reverse.t) < 1e-12)

ledger_rows = []
for step, row in enumerate(ledger_forward, start=1):
    roots = row["roots"]
    ledger_rows.append({
        "step": step,
        "surface": row["surface"],
        "candidate_t": "" if row["candidate_t"] is None else f"{row['candidate_t']:.8f}",
        "accepted_as_closest_so_far": row["accepted_as_closest_so_far"],
        "interval_t1_after_test": f"{row['interval_t1_after_test']:.8f}",
        "entry_root": "" if roots is None else f"{roots[0]:.8f}",
        "exit_root": "" if roots is None else f"{roots[1]:.8f}",
    })
ledger_path = save_table_csv(ledger_rows, TOPIC, "closest-hit-ledger.csv")
table_paths.append(ledger_path)

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.axhline(0, color="#d7dde5", lw=7, alpha=0.55)
colors = [PALETTE["blue"], PALETTE["gold"], PALETTE["green"]]
for idx, row in enumerate(ledger_forward):
    roots = row["roots"]
    if roots is None:
        continue
    y = 0.52 - idx * 0.32
    ax.plot([roots[0], roots[1]], [y, y], color=colors[idx], lw=8, solid_capstyle="round", label=row["surface"])
    ax.scatter([roots[0]], [y], color=colors[idx], edgecolor="white", s=90, zorder=4)
    ax.text(roots[0], y + 0.08, f"t={roots[0]:.2f}", ha="center", fontsize=8)
    if row["accepted_as_closest_so_far"]:
        ax.annotate("shrinks t1 here", xy=(roots[0], y), xytext=(roots[0] + 0.35, y + 0.25), arrowprops={"arrowstyle": "->", "color": PALETTE["red"]}, color=PALETTE["red"], fontsize=8)
ax.scatter([hit_forward.t], [0], marker="*", color=PALETTE["red"], s=150, zorder=5, label="returned closest hit")
style_axis(ax, "scene intersection as interval shrinking", xlabel="ray parameter t", ylabel="surface test order")
ax.set_yticks([])
ax.set_xlim(1.8, 6.2)
ax.legend(loc="upper right", fontsize=8)
add_note(ax, "Later tests only need hits closer than the current best t.")
closest_path = save_matplotlib(fig, TOPIC, "closest-hit-interval-shrink.png")
close(fig)
image_paths.append(closest_path)
closest_path


## 5. Shading Inputs, Shadows, And Mirror Rays

After a primary ray finds the visible surface, shading is just geometry plus a material rule. The hit point `x` comes from evaluating the ray at the winning `t`. The normal `n` belongs to the surface. A point light supplies a direction `l` from `x` to the light. The view direction `v` is the opposite of the incoming ray direction. These four quantities are enough to build the simple direct-light terms used here and in the next chapter.

Ray tracing makes shadows and mirror reflections natural because both are new rays. A shadow ray asks whether anything blocks the segment from the hit point to the light. A mirror ray asks what color is visible along the reflected direction. Both should start at a small positive epsilon instead of exactly at the surface.


In [ ]:
local_hit = hit_forward
light_position = np.array([2.6, 3.1, 0.7])
light_vec = light_position - local_hit.point
light_distance = float(np.linalg.norm(light_vec))
light_dir = unit(light_vec)
view_dir = -probe_ray.direction
reflection_dir = reflect(probe_ray.direction, local_hit.normal)
ndotl = float(max(0.0, np.dot(local_hit.normal, light_dir)))
checks["shading_vectors_unit_norms"] = {
    "normal": float(np.linalg.norm(local_hit.normal)),
    "light": float(np.linalg.norm(light_dir)),
    "view": float(np.linalg.norm(view_dir)),
    "reflection": float(np.linalg.norm(reflection_dir)),
}
checks["sample_ndotl"] = ndotl
checks["reflection_above_tangent_plane"] = bool(np.dot(reflection_dir, local_hit.normal) > 0)

dx, dy, dz, nx_s, ny_s, nz_s = sp.symbols("dx dy dz nx ny nz", real=True)
d = sp.Matrix([dx, dy, dz])
n = sp.Matrix([nx_s, ny_s, nz_s])
r = d - 2 * (d.dot(n)) * n
identity = sp.expand(r.dot(r) - d.dot(d))
identity_factored = sp.factor(identity)
expected_identity = sp.factor(4 * (d.dot(n))**2 * (n.dot(n) - 1))
assert sp.simplify(identity_factored - expected_identity) == 0
checks["reflection_identity_for_unit_normal"] = str(identity_factored)
checks["reflection_identity_unit_normal_condition"] = "identity is zero when nx**2 + ny**2 + nz**2 == 1"

fig, ax = plt.subplots(figsize=(8.8, 5.7))
sphere = scene_spheres[1]
theta = np.linspace(0, 2 * np.pi, 360)
cx, cz = sphere["center"][[0, 2]]
rad = sphere["radius"]
ax.fill(cx + rad * np.cos(theta), cz + rad * np.sin(theta), color="#f8d68a", alpha=0.58, ec=PALETTE["gold"], lw=1.7)
xz = lambda p: np.array([p[0], p[2]])
origin = xz(local_hit.point)
ax.scatter([origin[0]], [origin[1]], color=PALETTE["red"], s=60, zorder=5)
for vec, label, color in [
    (local_hit.normal, "n", PALETTE["green"]),
    (light_dir, "l", PALETTE["gold"]),
    (view_dir, "v", PALETTE["blue"]),
    (reflection_dir, "r", PALETTE["violet"]),
]:
    v2 = xz(vec)
    if np.linalg.norm(v2) < 1e-10:
        continue
    v2 = unit(v2) * 0.72
    ax.arrow(origin[0], origin[1], v2[0], v2[1], head_width=0.045, head_length=0.07, length_includes_head=True, color=color, lw=1.6)
    ax.text(origin[0] + v2[0] * 1.08, origin[1] + v2[1] * 1.08, label, color=color, fontsize=12, weight="bold")
ax.scatter([light_position[0]], [light_position[2]], marker="*", color=PALETTE["gold"], s=180, label="point light")
ax.plot([origin[0], light_position[0]], [origin[1], light_position[2]], color=PALETTE["gold"], lw=1.1, ls="--", alpha=0.8, label="shadow test segment")
style_axis(ax, "hit record geometry for shading, shadows, and reflection", equal=True, xlabel="world x", ylabel="world z")
ax.set_xlim(-1.6, 3.0)
ax.set_ylim(-4.1, 1.0)
ax.legend(fontsize=8, loc="upper left")
add_note(ax, f"sample max(0, n dot l) = {ndotl:.3f}; reflection preserves direction length")
shade_path = save_matplotlib(fig, TOPIC, "shading-shadow-reflection-rays.png")
close(fig)
image_paths.append(shade_path)
shade_path


In [ ]:
self_hit_t_values = np.geomspace(2e-7, 1.2e-3, 28)
close_blocker_t = 0.035
eps_values = np.geomspace(1e-8, 1e-1, 260)
self_acne_counts = np.array([np.count_nonzero(eps <= self_hit_t_values) for eps in eps_values])
misses_close_blocker = eps_values >= close_blocker_t
safe_mask = (self_acne_counts == 0) & (~misses_close_blocker)
safe_min = float(eps_values[np.where(safe_mask)[0][0]])
safe_max = float(eps_values[np.where(safe_mask)[0][-1]])
checks["epsilon_safe_band"] = {"min": safe_min, "max": safe_max, "close_blocker_t": close_blocker_t}
checks["epsilon_self_hit_max"] = float(self_hit_t_values.max())
checks["epsilon_safe_band_exists"] = bool(np.any(safe_mask))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
ax = axes[0]
ax.loglog(eps_values, self_acne_counts + 1e-12, color=PALETTE["red"], lw=2.0)
ax.axvline(self_hit_t_values.max(), color=PALETTE["green"], ls="--", lw=1.5, label="larger than all self-hit t values")
ax.axvline(close_blocker_t, color=PALETTE["violet"], ls="--", lw=1.5, label="close blocker distance")
style_axis(ax, "shadow acne count as epsilon grows", xlabel="epsilon used as ray t0", ylabel="number of self hits not skipped")
ax.legend(fontsize=8)
add_note(ax, "epsilon must exceed tiny self-hit roots but remain below real blocker distances")

ax = axes[1]
ax.semilogx(eps_values, safe_mask.astype(int), color=PALETTE["blue"], lw=2.2)
ax.fill_between(eps_values, 0, safe_mask.astype(float), color=PALETTE["blue"], alpha=0.18)
ax.set_yticks([0, 1])
ax.set_yticklabels(["bad", "usable"])
style_axis(ax, "valid epsilon band for this scale", xlabel="epsilon", ylabel="classification")
add_note(ax, f"usable roughly from {safe_min:.1e} to {safe_max:.1e}")

epsilon_path = save_matplotlib(fig, TOPIC, "epsilon-shadow-acne-pitfall.png")
close(fig)
image_paths.append(epsilon_path)
epsilon_path


## Applied lab: render a small scene and change reflection depth

The lab below is deliberately simple: two spheres, a floor plane, one point light, ambient fill, shadows, and optional mirror reflection. It is not meant to be a production renderer. Its purpose is to make the Chapter 04 control flow executable: primary rays find closest hits; direct shading uses hit-record data; shadow rays test a finite segment to the light; mirror rays recurse only while the remaining depth is positive.

Inspect the two panels after running the cell. With depth zero, the image contains direct lighting and shadows only. With depth two, the reflective floor and mirror-weighted sphere add secondary-ray color, but the recursion stops at the chosen limit.


In [ ]:
@dataclass(frozen=True)
class SphereObject:
    name: str
    center: np.ndarray
    radius: float
    color: np.ndarray
    mirror: np.ndarray

    def hit(self, ray: Ray, t0: float, t1: float) -> Hit | None:
        return sphere_hit(ray, self.center, self.radius, self.name, self.color, self.mirror, t0=t0, t1=t1)


@dataclass(frozen=True)
class PlaneObject:
    name: str
    point: np.ndarray
    normal: np.ndarray
    color: np.ndarray
    mirror: np.ndarray

    def hit(self, ray: Ray, t0: float, t1: float) -> Hit | None:
        denom = float(np.dot(ray.direction, self.normal))
        if abs(denom) < 1e-10:
            return None
        t = float(np.dot(self.point - ray.origin, self.normal) / denom)
        if not (t0 <= t <= t1):
            return None
        point = ray.at(t)
        normal = self.normal if denom < 0 else -self.normal
        checker = (math.floor(point[0] * 1.2) + math.floor(point[2] * 1.2)) % 2
        color = self.color * (0.72 if checker else 1.0)
        return Hit(t, point, unit(normal), self.name, color, self.mirror)


render_camera = make_camera(
    eye=np.array([0.0, 0.75, 2.8]),
    look_at=np.array([0.0, -0.2, -2.8]),
    up=np.array([0.0, 1.0, 0.0]),
    left=-1.35,
    right=1.35,
    bottom=-0.78,
    top=0.78,
    distance=1.1,
)
objects = [
    SphereObject("matte blue sphere", np.array([-0.62, -0.18, -2.65]), 0.58, np.array([0.22, 0.42, 0.86]), np.array([0.03, 0.04, 0.08])),
    SphereObject("mirror rose sphere", np.array([0.48, -0.28, -3.35]), 0.72, np.array([0.9, 0.42, 0.36]), np.array([0.38, 0.27, 0.25])),
    SphereObject("small shadow caster", np.array([0.15, 0.43, -2.05]), 0.18, np.array([0.34, 0.74, 0.55]), np.array([0.0, 0.0, 0.0])),
    PlaneObject("mirror floor", np.array([0.0, -0.85, 0.0]), np.array([0.0, 1.0, 0.0]), np.array([0.78, 0.80, 0.76]), np.array([0.22, 0.22, 0.22])),
]
light_pos = np.array([2.4, 3.4, 1.6])
light_rgb = np.array([1.0, 0.95, 0.86])
ambient_rgb = np.array([0.065, 0.075, 0.09])
background_top = np.array([0.74, 0.84, 0.98])
background_bottom = np.array([0.95, 0.97, 1.0])
EPS = 1e-4
hit_counter = {"primary_hits": 0, "shadow_blocked": 0, "reflection_rays": 0}


def scene_hit(ray: Ray, *, t0=EPS, t1=np.inf) -> Hit | None:
    closest = None
    closest_t = float(t1)
    for obj in objects:
        hit = obj.hit(ray, t0, closest_t)
        if hit is not None and hit.t < closest_t:
            closest = hit
            closest_t = hit.t
    return closest


def background(ray: Ray) -> np.ndarray:
    amount = 0.5 * (unit(ray.direction)[1] + 1.0)
    return (1.0 - amount) * background_bottom + amount * background_top


def shade(ray: Ray, depth: int) -> np.ndarray:
    hit = scene_hit(ray, t0=EPS, t1=80.0)
    if hit is None:
        return background(ray)
    hit_counter["primary_hits"] += 1
    x = hit.point
    n = hit.normal
    to_light = light_pos - x
    light_distance = float(np.linalg.norm(to_light))
    l = to_light / light_distance
    shadow_ray = Ray(x + EPS * l, l)
    blocker = scene_hit(shadow_ray, t0=EPS, t1=light_distance - EPS)
    if blocker is None:
        ndotl = max(0.0, float(np.dot(n, l)))
        direct = hit.color * light_rgb * ndotl / (0.18 * light_distance * light_distance)
    else:
        hit_counter["shadow_blocked"] += 1
        direct = np.zeros(3)
    local = hit.color * ambient_rgb + direct
    if depth <= 0 or np.max(hit.mirror) <= 0:
        return np.clip(local, 0.0, 1.0)
    hit_counter["reflection_rays"] += 1
    r = reflect(ray.direction, n)
    reflected = shade(Ray(x + EPS * r, r), depth - 1)
    return np.clip((1.0 - np.max(hit.mirror) * 0.45) * local + hit.mirror * reflected, 0.0, 1.0)


def render_scene(width=176, height=108, depth=0) -> np.ndarray:
    image = np.zeros((height, width, 3), dtype=float)
    for j in range(height):
        for i in range(width):
            ray = perspective_ray(i, height - 1 - j, width, height, render_camera)
            image[j, i] = shade(ray, depth)
    return np.clip(image, 0.0, 1.0)

hit_counter = {"primary_hits": 0, "shadow_blocked": 0, "reflection_rays": 0}
image_depth0 = render_scene(depth=0)
counters_depth0 = dict(hit_counter)
hit_counter = {"primary_hits": 0, "shadow_blocked": 0, "reflection_rays": 0}
image_depth2 = render_scene(depth=2)
counters_depth2 = dict(hit_counter)
image_difference = float(np.mean(np.abs(image_depth2 - image_depth0)))
checks["render_depth0"] = counters_depth0
checks["render_depth2"] = counters_depth2
checks["reflection_depth_image_mean_abs_difference"] = image_difference
checks["render_image_shape"] = list(image_depth2.shape)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.2))
for ax, image, title in [(axes[0], image_depth0, "depth 0: direct light + shadows"), (axes[1], image_depth2, "depth 2: mirror rays included")]:
    ax.imshow(image)
    ax.set_title(title, fontsize=11, color=PALETTE["ink"])
    ax.set_xticks([])
    ax.set_yticks([])
add_note(axes[0], f"blocked shadow tests: {counters_depth0['shadow_blocked']}")
add_note(axes[1], f"reflection rays: {counters_depth2['reflection_rays']}")
reflection_path = save_matplotlib(fig, TOPIC, "reflection-depth-comparison.png")
close(fig)
image_paths.append(reflection_path)

fig, ax = plt.subplots(figsize=(6.4, 4.1))
ax.imshow(image_depth2)
ax.set_title("small recursive ray traced scene", fontsize=11, color=PALETTE["ink"])
ax.set_xticks([])
ax.set_yticks([])
add_note(ax, "The rendered image is produced by the same ray, hit, shadow, and reflection functions checked above.")
scene_path = save_matplotlib(fig, TOPIC, "mini-ray-traced-scene.png")
close(fig)
image_paths.append(scene_path)
(scene_path, reflection_path)


## Display The Chapter Artifacts

Every artifact below was generated from the code in this notebook and saved under `artifacts/chapter-04`. The file names state the concept each artifact teaches. The PNGs are durable static views for the course build, the HTML file preserves an interactive 3D camera-ray fan, the CSV file records the closest-hit ledger, and the JSON files store reproducible checks.


In [ ]:
artifact_sequence = [*image_paths, *html_paths, *table_paths, *check_paths]
assert_artifacts(artifact_sequence)
for path in image_paths:
    display_artifact(path, width=760)
for path in html_paths:
    display_artifact(path, width="100%", height=560)
for path in [*table_paths, *check_paths]:
    display_artifact(path)


## Sanity checks

These checks are tied to Chapter 04 implementation hazards. A passing notebook must show that the camera frame is orthonormal, the center perspective ray follows the view direction, the sphere roots are ordered and correct for a simple center ray, the triangle hit has valid barycentric coordinates, closest-hit selection does not depend on object-list order, shading vectors are normalized, reflection preserves direction length, an epsilon band exists for the chosen scene scale, recursive reflection changes the rendered image, and all artifacts are present and nonblank.


In [ ]:
assert checks["camera_frame_orthonormal_error"] < 1e-12
assert checks["camera_center_ray_passes_optical_axis"] is True
assert np.allclose(checks["sphere_center_roots"], [3.0, 5.0], atol=1e-12)
assert np.allclose(checks["sphere_center_hit_normal"], [0.0, 0.0, 1.0], atol=1e-12)
bar = checks["triangle_debug_barycentric"]
assert abs(bar["sum"] - 1.0) < 1e-12
assert min(bar["alpha"], bar["beta"], bar["gamma"]) >= -1e-12
assert checks["triangle_plane_z_residual"] < 1e-12
assert checks["closest_hit_order_invariant"] is True
assert checks["closest_hit_name"] == "front gold sphere"
assert abs(checks["shading_vectors_unit_norms"]["normal"] - 1.0) < 1e-12
assert abs(checks["shading_vectors_unit_norms"]["light"] - 1.0) < 1e-12
assert checks["reflection_above_tangent_plane"] is True
assert checks["epsilon_safe_band_exists"] is True
assert checks["reflection_depth_image_mean_abs_difference"] > 0.005

artifact_records = assert_artifacts([*image_paths, *html_paths, *table_paths, *check_paths])
image_records = [assert_nonblank_image(path) for path in image_paths]
for record in image_records:
    record["path"] = book_relative(record["path"])
checks["artifact_count_before_final_json"] = len(artifact_records)
checks["nonblank_png_count"] = len(image_records)
checks["artifact_records"] = artifact_records
checks["image_records"] = image_records
numeric_path = save_json(checks, TOPIC, "ray-tracing-invariants.json")
legacy_numeric_path = save_json(checks, TOPIC, "numeric-checks.json")
check_paths.extend([numeric_path, legacy_numeric_path])
final_report = {
    "chapter": CHAPTER,
    "title": "Ray Tracing",
    "printed_pages": PRINTED_PAGES,
    "pdf_pages": PDF_PAGES,
    "checks_passed": True,
    "nonblank_png_count": len(image_records),
    "core_checks": {
        "camera_center_ray_alignment": checks["center_ray_alignment_with_view"],
        "sphere_center_roots": checks["sphere_center_roots"],
        "triangle_barycentric_sum": bar["sum"],
        "closest_hit_name": checks["closest_hit_name"],
        "epsilon_safe_band": checks["epsilon_safe_band"],
        "reflection_depth_difference": checks["reflection_depth_image_mean_abs_difference"],
    },
}
final_path = save_json(final_report, TOPIC, "final-sanity.json")
check_paths.append(final_path)
final_report["artifact_count"] = len(assert_artifacts([*image_paths, *html_paths, *table_paths, *check_paths]))
final_path = save_json(final_report, TOPIC, "final-sanity.json")
display_artifact(numeric_path)
display_artifact(final_path)
final_report


## Takeaways

Ray tracing starts as a simple image-order loop, but the correctness lives in the small geometric contracts: pixel centers map into a camera frame, rays carry intervals, surfaces return hit records rather than bare booleans, and the group intersection keeps the nearest valid `t`.

Perspective needs no projection matrix in this setting because the projection is implicit in the fan of primary rays from the eye through the image plane. Shadows and mirror reflection are natural extensions of the same ray query, provided secondary rays use a scale-aware epsilon and reflection recursion has a depth limit.

The final renderer in this notebook is intentionally compact. Its value is that every image feature can be traced back to an inspectable quantity: a root, a barycentric coordinate, a normal, a dot product, a shadow interval, or a reflected direction.
